### Data Aquisition 

Purpose: Ingest, clean, and transform the Berkeley Carbon Trading Project database into a relational $D_1$ dataset for DS 4320.

Data Source: Berkeley Carbon Trading Project (GSPP)

Date Acquired: March 25, 2026

Output: * 5 CSV Tables (~3+ MB total) and 5 Parquet Files (~1.2+ MB total) down from the original ~200 MB raw data

Relational Structure: 
- `developers` (1:N via `projects.developer_id`)
- `projects` (core)
- `locations`, `methodology`, `performance` (each with unique PKs and `project_id` as FK)

In [2]:
import logging
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

# 1. SETUP DIRECTORIES
# Assumption: you run this notebook from inside the repo's `code/` folder.
CODE_DIR = Path(os.getcwd())
BASE_DIR = CODE_DIR.parent
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. SETUP LOGGING
# In Jupyter, logging may already be configured; `force=True` guarantees our FileHandler is attached.
LOG_PATH = CODE_DIR / "carbon_acquisition.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(),
    ],
    force=True,
)

print("=" * 70)
print("BERKELEY CARBON TRADING PROJECT: FLAT DATA PIPELINE")
print("=" * 70)


def ensure_fastparquet() -> None:
    """Ensure `fastparquet` is available so Parquet writing doesn't rely on pyarrow.

    This avoids the common pandas/pyarrow conflict: `A type extension with name pandas.period already defined`.
    """

    try:
        import fastparquet  # noqa: F401
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "fastparquet"]
        )


def process_carbon_data() -> dict[str, pd.DataFrame] | None:
    """Load the raw Berkeley offsets CSV and split it into 5 relational tables.

    Tables (3NF-friendly):
    - `developers` provides a 1:N reference for `projects.developer_id`
    - `projects` has PK `project_id` (source ID)
    - `locations`, `methodology`, `performance` each get their own surrogate PKs
      while still relating back via FK `project_id`.
    """

    raw_path = DATA_DIR / "berkeley_carbon_raw.csv"
    if not raw_path.exists():
        logging.error(
            "Raw file not found at %s. Put `berkeley_carbon_raw.csv` in /data.",
            raw_path,
        )
        return None

    logging.info("Ingesting raw Berkeley dataset from %s", raw_path)
    df = pd.read_csv(raw_path, skiprows=3, low_memory=False)
    logging.info("Raw CSV loaded: rows=%s cols=%s", len(df), len(df.columns))

    # Cleaning
    df.columns = df.columns.str.replace("\n", " ").str.strip()

    before_cols = len(df.columns)
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
    after_cols = len(df.columns)
    logging.info(
        "Dropped Unnamed columns: %s removed (%s → %s)",
        before_cols - after_cols,
        before_cols,
        after_cols,
    )

    # Normalize developer strings (helps keep stable IDs across runs)
    df["Project Developer"] = (
        df["Project Developer"].fillna("Unknown").astype(str).str.strip()
    )

    # ----- Dimension: developers (stable PK generation) -----
    developers = (
        df[["Project Developer"]]
        .drop_duplicates(subset=["Project Developer"])
        .rename(columns={"Project Developer": "developer"})
        .sort_values("developer")
        .reset_index(drop=True)
    )
    developers["developer_id"] = developers.index + 1
    logging.info("Developers table built: rows=%s", len(developers))

    # Attach developer_id to each row for downstream PK/FK wiring
    df = df.merge(
        developers[["developer", "developer_id"]],
        left_on="Project Developer",
        right_on="developer",
        how="left",
    )
    df = df.drop(columns=["developer"])

    # ----- Core: projects (PK = project_id, FK = developer_id) -----
    projects = df[
        [
            "Project ID",
            "Project Name",
            "Project Type From the Registry",
            "developer_id",
        ]
    ].copy()
    projects.columns = ["project_id", "name", "type", "developer_id"]
    projects = projects.sort_values("project_id").reset_index(drop=True)
    logging.info("Projects table built: rows=%s", len(projects))

    # ----- Dimensions with surrogate PKs -----
    locations = df[["Project ID", "Region", "Country", "State", "Project Site Location"]].copy()
    locations.columns = ["project_id", "region", "country", "state", "site_location"]
    locations = locations.sort_values(
        ["project_id", "region", "country", "state", "site_location"]
    ).reset_index(drop=True)
    locations["location_id"] = locations.index + 1
    logging.info("Locations table built: rows=%s", len(locations))

    methodology = df[["Project ID", "Voluntary Registry", "Methodology / Protocol", "Scope", "Type"]].copy()
    methodology.columns = ["project_id", "registry", "protocol", "scope", "type"]
    methodology = methodology.sort_values(
        ["project_id", "registry", "protocol", "scope", "type"]
    ).reset_index(drop=True)
    methodology["methodology_id"] = methodology.index + 1
    logging.info("Methodology table built: rows=%s", len(methodology))

    performance = df[
        [
            "Project ID",
            "Total Credits  Issued",
            "Total Credits  Retired",
            "Total Credits Remaining",
            "First Year of Project (Vintage)",
        ]
    ].copy()
    performance.columns = ["project_id", "issued", "retired", "remaining", "vintage_year"]
    performance = performance.sort_values(
        ["project_id", "issued", "retired", "remaining", "vintage_year"]
    ).reset_index(drop=True)
    performance["performance_id"] = performance.index + 1
    logging.info("Performance table built: rows=%s", len(performance))

    # Ensure PKs are first columns for readability
    developers = developers[["developer_id", "developer"]]
    projects = projects[["project_id", "name", "developer_id", "type"]]
    locations = locations[["location_id", "project_id", "region", "country", "state", "site_location"]]
    methodology = methodology[["methodology_id", "project_id", "registry", "protocol", "scope", "type"]]
    performance = performance[["performance_id", "project_id", "issued", "retired", "remaining", "vintage_year"]]

    return {
        "developers": developers,
        "projects": projects,
        "locations": locations,
        "methodology": methodology,
        "performance": performance,
    }


def save_and_verify(tables: dict[str, pd.DataFrame]) -> None:
    """Save each table to CSV + Parquet and print rubric-oriented checks."""

    ensure_fastparquet()

    logging.info("Saving %s tables to %s", len(tables), DATA_DIR)
    print("\n[SAVING TO DATA FOLDER]")

    total_csv_bytes = 0
    total_parquet_bytes = 0

    for name, table_df in tables.items():
        csv_path = DATA_DIR / f"{name}.csv"
        pq_path = DATA_DIR / f"{name}.parquet"

        # CSV (always works)
        table_df.to_csv(csv_path, index=False)

        # Parquet (use fastparquet to avoid pyarrow extension registration conflicts)
        table_df.to_parquet(
            pq_path,
            index=False,
            engine="fastparquet",
            compression="gzip",
        )

        csv_bytes = os.path.getsize(csv_path)
        pq_bytes = os.path.getsize(pq_path)

        total_csv_bytes += csv_bytes
        total_parquet_bytes += pq_bytes

        logging.info(
            "Saved %s: rows=%s csv=%s (%.2f MB) parquet=%s (%.2f MB)",
            name,
            len(table_df),
            csv_path,
            csv_bytes / (1024 * 1024),
            pq_path,
            pq_bytes / (1024 * 1024),
        )

        print(
            f"  ✓ {name:12} | Rows: {len(table_df):<7} | Saved: CSV + Parquet"
        )

    total_mb = total_csv_bytes / (1024 * 1024)
    logging.info(
        "Write complete: csv_total=%.2f MB parquet_total=%.2f MB",
        total_csv_bytes / (1024 * 1024),
        total_parquet_bytes / (1024 * 1024),
    )

    print("\n" + "=" * 70)
    print("RUBRIC CHECKLIST")
    print(f"✓ Minimum 5 Tables: {len(tables)}")
    print("✓ PKs: project_id (projects), developer_id (developers), plus surrogate IDs in dimensions")
    print("✓ FKs: projects.developer_id → developers.developer_id; dimensions.project_id → projects.project_id")
    print("✓ CSV: written")
    print("✓ Parquet: written")
    print(f"✓ Scale (CSV total): {total_mb:.2f} MB")
    print("=" * 70)


# EXECUTE
try:
    logging.info("Pipeline starting. Log file: %s", LOG_PATH)
    processed_tables = process_carbon_data()
    if processed_tables:
        save_and_verify(processed_tables)
        logging.info("Pipeline completed successfully.")
except Exception as e:
    logging.exception("Pipeline failed.")
    raise


2026-03-30 22:19:05,240 - INFO - Pipeline starting. Log file: /Users/oliverandress/ds4320-project1/code/carbon_acquisition.log
2026-03-30 22:19:05,244 - INFO - Ingesting raw Berkeley dataset from /Users/oliverandress/ds4320-project1/data/berkeley_carbon_raw.csv


BERKELEY CARBON TRADING PROJECT: FLAT DATA PIPELINE


2026-03-30 22:19:14,271 - INFO - Raw CSV loaded: rows=11473 cols=16268
2026-03-30 22:19:14,313 - INFO - Dropped Unnamed columns: 16098 removed (16268 → 170)
2026-03-30 22:19:14,327 - INFO - Developers table built: rows=3676
2026-03-30 22:19:14,363 - INFO - Projects table built: rows=11473
2026-03-30 22:19:14,372 - INFO - Locations table built: rows=11473
2026-03-30 22:19:14,380 - INFO - Methodology table built: rows=11473
2026-03-30 22:19:14,393 - INFO - Performance table built: rows=11473
2026-03-30 22:19:14,404 - INFO - Saving 5 tables to /Users/oliverandress/ds4320-project1/data
2026-03-30 22:19:14,425 - INFO - Saved developers: rows=3676 csv=/Users/oliverandress/ds4320-project1/data/developers.csv (0.13 MB) parquet=/Users/oliverandress/ds4320-project1/data/developers.parquet (0.05 MB)
2026-03-30 22:19:14,485 - INFO - Saved projects: rows=11473 csv=/Users/oliverandress/ds4320-project1/data/projects.csv (1.06 MB) parquet=/Users/oliverandress/ds4320-project1/data/projects.parquet (0.2


[SAVING TO DATA FOLDER]
  ✓ developers   | Rows: 3676    | Saved: CSV + Parquet
  ✓ projects     | Rows: 11473   | Saved: CSV + Parquet
  ✓ locations    | Rows: 11473   | Saved: CSV + Parquet
  ✓ methodology  | Rows: 11473   | Saved: CSV + Parquet


2026-03-30 22:19:14,625 - INFO - Saved performance: rows=11473 csv=/Users/oliverandress/ds4320-project1/data/performance.csv (0.35 MB) parquet=/Users/oliverandress/ds4320-project1/data/performance.parquet (0.11 MB)
2026-03-30 22:19:14,625 - INFO - Write complete: csv_total=3.30 MB parquet_total=0.56 MB
2026-03-30 22:19:14,625 - INFO - Pipeline completed successfully.


  ✓ performance  | Rows: 11473   | Saved: CSV + Parquet

RUBRIC CHECKLIST
✓ Minimum 5 Tables: 5
✓ PKs: project_id (projects), developer_id (developers), plus surrogate IDs in dimensions
✓ FKs: projects.developer_id → developers.developer_id; dimensions.project_id → projects.project_id
✓ CSV: written
✓ Parquet: written
✓ Scale (CSV total): 3.30 MB
